## DATA-CORPUS

In [1]:
import os
import pandas as pd
import pdfplumber
from docx import Document
import re
from datetime import datetime

# Set working directory
os.chdir("/Users/cansezgin/Library/Mobile Documents/com~apple~CloudDocs/Claude_Projects/Computational Social Sciences/conflict-text-analysis")

# Verify we can see the raw data
raw_files = os.listdir("data/raw")
print(f"Found {len(raw_files)} files in data/raw/:")
for f in sorted(raw_files):
    print(f"  {f}")

Found 18 files in data/raw/:
  OHCHR-EHRC-Tigray-Report.pdf
  SC:14501 (April 22, 2021). Security Council press statement on Ethiopia.docx
  SG:SM:20563, 2021-02-02..docx
  SG:SM:20798, 2021-06-25. "Strongly Condemning Air Strike.docx
  SG:SM:20799, 2021-06-26. "Welcoming Demonstrable Effort.docx
  SG:SM:20804, 2021-06-28. "Recent Events in Ethiopia's Tigray Region 'Extremely Worrisome'.docx
  SG:SM:20821, 2021-07-09. "Secretary-General Deeply Shocked by Killing of Humanitarian Workers.docx
  SG:SM:21109, 2022-01-19.docx
  SG:SM:21232, 2022-04-06. "Welcoming News of Food Aid Reaching Ethiopia's Tigray, Afar.docx
  SG:SM:21444, 2022-09-11. "Secretary-General, Concerned about Situation.docx
  SG:SM:21534, 2022-10-15. "Secretary-General Gravely Concerned by Escalation.docx
  SG:SM:21566, 2022-11-02. "Secretary-General Welcomes Signing of Peace Agreement.docx
  S_2021_378-EN.pdf
  S_2021_510-EN.pdf
  S_PV.8812-EN.pdf
  S_PV.8843-EN.pdf
  S_PV.8875-EN.pdf
  S_PV.8899-EN.pdf


In [2]:
# CELL 2: Extract text from all PDF files

def extract_pdf_text(filepath):
    """Extract all text from a PDF file using pdfplumber."""
    text = ""
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text.strip()

# Extract text from all PDFs
pdf_files = [f for f in raw_files if f.endswith(".pdf")]
pdf_texts = {}

for f in sorted(pdf_files):
    filepath = os.path.join("data/raw", f)
    text = extract_pdf_text(filepath)
    pdf_texts[f] = text
    print(f"{f}: {len(text)} characters, {len(text.split())} words")

OHCHR-EHRC-Tigray-Report.pdf: 410915 characters, 62658 words
S_2021_378-EN.pdf: 3226 characters, 504 words
S_2021_510-EN.pdf: 5209 characters, 811 words
S_PV.8812-EN.pdf: 86762 characters, 13554 words
S_PV.8843-EN.pdf: 78007 characters, 12116 words
S_PV.8875-EN.pdf: 75021 characters, 11612 words
S_PV.8899-EN.pdf: 72483 characters, 11393 words


In [3]:
# CELL 3: Extract text from all DOCX files

def extract_docx_text(filepath):
    """Extract all text from a DOCX file."""
    doc = Document(filepath)
    text = "\n".join([para.text for para in doc.paragraphs if para.text.strip()])
    return text.strip()

# Extract text from all DOCX files
docx_files = [f for f in raw_files if f.endswith(".docx")]
docx_texts = {}

for f in sorted(docx_files):
    filepath = os.path.join("data/raw", f)
    text = extract_docx_text(filepath)
    docx_texts[f] = text
    print(f"{f}: {len(text)} characters, {len(text.split())} words")

SC:14501 (April 22, 2021). Security Council press statement on Ethiopia.docx: 2325 characters, 334 words
SG:SM:20563, 2021-02-02..docx: 1292 characters, 178 words
SG:SM:20798, 2021-06-25. "Strongly Condemning Air Strike.docx: 864 characters, 128 words
SG:SM:20799, 2021-06-26. "Welcoming Demonstrable Effort.docx: 633 characters, 89 words
SG:SM:20804, 2021-06-28. "Recent Events in Ethiopia's Tigray Region 'Extremely Worrisome'.docx: 572 characters, 86 words
SG:SM:20821, 2021-07-09. "Secretary-General Deeply Shocked by Killing of Humanitarian Workers.docx: 1139 characters, 148 words
SG:SM:21109, 2022-01-19.docx: 1974 characters, 309 words
SG:SM:21232, 2022-04-06. "Welcoming News of Food Aid Reaching Ethiopia's Tigray, Afar.docx: 858 characters, 123 words
SG:SM:21444, 2022-09-11. "Secretary-General, Concerned about Situation.docx: 1155 characters, 174 words
SG:SM:21534, 2022-10-15. "Secretary-General Gravely Concerned by Escalation.docx: 683 characters, 98 words
SG:SM:21566, 2022-11-02. "S

In [4]:
# CELL 4: Inspect the structure of a verbatim record
# Look at the first 3000 characters of S/PV.8812

sample = pdf_texts["S_PV.8812-EN.pdf"]
print(sample[:3000])

S
United Nations
/PV.8812
Security Council
Provisional
Seventy-sixth year
8812
th meeting
Friday, 2 July 2021, 3 p.m.
New York
President: Mr. De Rivière .................................. (France)
Members: China ......................................... Mr. Dai Bing
Estonia ........................................ Mr. Jürgenson
India ......................................... Mr. Tirumurti
Ireland ........................................ Ms. Byrne Nason
Kenya. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . Mr. Kimani
Mexico ........................................ Mrs. Buenrostro Massieu
Niger ......................................... Mr. Maman Sani
Norway ....................................... Ms. Juul
Russian Federation ............................... Mr. Nebenzia
Saint Vincent and the Grenadines ................... Ms. King
Tunisia ........................................ Mr. Ladeb
United Kingdom of Great Britain and Northern Ireland .. Lord Ahmad
Un

In [5]:
# CELL 5: Look at how speeches begin in the document
# Search for patterns like "Mr." or "Mrs." that start speeches

sample = pdf_texts["S_PV.8812-EN.pdf"]
print(sample[2000:5000])

s to the outgoing President
The situation in Mekelle is reportedly calm, and
The President (spoke in French): As this is the the TDF appear to be in control of the city. Reports
first public meeting of the Security Council for the indicate that leaders of the previous Tigray regional
month of July, I should like to take this opportunity to administration, including its former president, have
pay tribute, on behalf of the Council, to His Excellency returned to Mekelle. As of today, the TDF has yet to
Mr. Sven Jürgenson, Permanent Representative of agree to a ceasefire.
Estonia, for his service as President of the Security
While there have been no reports of serious
Council for the month of June. I am sure I speak for
incidents, basic services to support humanitarian
all the members of the Council in expressing deep
delivery are absent. Mekelle has no electrical power
appreciation to Ambassador Jürgenson and his team for
or Internet. Key infrastructure has been destroyed,
the great diplo

In [6]:
# CELL 6: Look further into the document for member state speeches

sample = pdf_texts["S_PV.8812-EN.pdf"]
print(sample[10000:14000])

 both challenging and hopeful. It has
will. Peace and stability in the country, the cornerstone
brought to the fore disagreements around fundamental
of the Horn region, may well depend on it.
issues such as the federal structure of the State and the
role and status of ethnicity, as well as how such disputes Allow me to offer some areas of concerted
should be addressed. international support to Ethiopia as it traverses the
current crisis.
The recent national elections were an important
milestone in that regard. They were, by many accounts, The international community must continue to call
an improvement on previous polls in the country for a permanent ceasefire to be honoured by all parties.
and were held in a generally peaceful manner. They We should urge Ethiopia’s leaders to work swiftly to
were, however, affected by insecurity and technical restore national unity through a process of inclusive
21-17867 3/19
S/PV.8812 Peace and security in Africa 02/07/2021
dialogue and reconciliatio

In [7]:
# CELL 7: Search for speaker transition patterns

sample = pdf_texts["S_PV.8812-EN.pdf"]

# Find lines that look like speaker introductions
lines = sample.split("\n")
for i, line in enumerate(lines):
    if any(phrase in line for phrase in ["I now give the floor", "The President:", "spoke in"]):
        # Print this line and the next few for context
        print(f"Line {i}: {line[:150]}")
        print()

Line 41: The President (spoke in French): As this is the the TDF appear to be in control of the city. Reports

Line 65: The President (spoke in French): In accordance

Line 90: I now give the floor to Ms. DiCarlo. Food insecurity has continued only to worsen in recent

Line 210: The President (spoke in French): I thank

Line 214: I now give the floor to Mr. Rajasingham. attacks are prohibited. Allegations of serious violations

Line 392: The President (spoke in French): I thank

Line 396: I now give the floor to those Council members who

Line 778: Mrs. Buenrostro Massieu (Mexico) (spoke in the population are being targeted, which includes entering

Line 820: hospital infrastructure is at risk of collapse, thereby Mr. Dai Bing (China) (spoke in Chinese): At the

Line 952: Mr. Nebenzia (Russian Federation) (spoke in

Line 1167: The President (spoke in French): I shall now



In [8]:
# CELL 8: Parse verbatim records into individual speeches

def parse_speeches(text, document_id, date):
    """
    Parse a UNSC verbatim record into individual speeches.
    Returns a list of dicts with speaker, country, speech text, etc.
    """
    # Pattern matches speaker introductions like:
    # Mr. Dai Bing (China) (spoke in Chinese):
    # Mrs. Thomas-Greenfield (United States of America):
    # The President (spoke in French):
    # Ms. DiCarlo:
    # Lord Ahmad:
    speaker_pattern = re.compile(
        r'((?:Mr\.|Mrs\.|Ms\.|Lord|Sir|The President)'  # Title
        r'[^:]{0,120}'  # Name and country (up to 120 chars)
        r')\s*:'         # Colon after name
    )
    
    matches = list(speaker_pattern.finditer(text))
    
    speeches = []
    for i, match in enumerate(matches):
        speaker_raw = match.group(1).strip()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        speech_text = text[start:end].strip()
        
        # Skip very short segments (procedural remarks)
        if len(speech_text.split()) < 30:
            continue
        
        # Extract country if present (text in first parentheses)
        country_match = re.search(r'\(([^)]+)\)', speaker_raw)
        country = ""
        if country_match:
            candidate = country_match.group(1)
            # Filter out "spoke in ..." which is not a country
            if not candidate.startswith("spoke"):
                country = candidate
        
        # Clean speaker name: take text before first parenthesis
        speaker_name = re.split(r'\s*\(', speaker_raw)[0].strip()
        
        speeches.append({
            "document_id": document_id,
            "date": date,
            "speaker_raw": speaker_raw,
            "speaker_name": speaker_name,
            "country": country,
            "word_count": len(speech_text.split()),
            "text": speech_text
        })
    
    return speeches

# Parse all 4 verbatim records
pv_metadata = {
    "S_PV.8812-EN.pdf": ("S/PV.8812", "2021-07-02"),
    "S_PV.8843-EN.pdf": ("S/PV.8843", "2021-08-26"),
    "S_PV.8875-EN.pdf": ("S/PV.8875", "2021-10-06"),
    "S_PV.8899-EN.pdf": ("S/PV.8899", "2021-11-08"),
}

all_speeches = []

for filename, (doc_id, date) in pv_metadata.items():
    text = pdf_texts[filename]
    speeches = parse_speeches(text, doc_id, date)
    all_speeches.extend(speeches)
    print(f"{doc_id} ({date}): {len(speeches)} speeches extracted")

print(f"\nTotal speeches extracted: {len(all_speeches)}")

S/PV.8812 (2021-07-02): 20 speeches extracted
S/PV.8843 (2021-08-26): 17 speeches extracted
S/PV.8875 (2021-10-06): 18 speeches extracted
S/PV.8899 (2021-11-08): 20 speeches extracted

Total speeches extracted: 75


In [9]:
# CELL 9: Inspect the extracted speeches

df_speeches = pd.DataFrame(all_speeches)

# Show speaker distribution
print("=== SPEECHES PER MEETING ===")
print(df_speeches.groupby("document_id").size())
print()

print("=== WORD COUNT STATS ===")
print(df_speeches["word_count"].describe().round(0))
print()

print("=== SAMPLE: First 5 speeches (speaker, country, word count) ===")
for i, row in df_speeches.head(5).iterrows():
    print(f"  {row['speaker_name']:30s} | {row['country']:35s} | {row['word_count']} words")
print()

print("=== ALL UNIQUE SPEAKERS ===")
for _, row in df_speeches[["speaker_name", "country"]].drop_duplicates().sort_values("country").iterrows():
    print(f"  {row['speaker_name']:30s} | {row['country']}")

=== SPEECHES PER MEETING ===
document_id
S/PV.8812    20
S/PV.8843    17
S/PV.8875    18
S/PV.8899    20
dtype: int64

=== WORD COUNT STATS ===
count      75.0
mean      642.0
std       442.0
min        35.0
25%       274.0
50%       539.0
75%       976.0
max      1776.0
Name: word_count, dtype: float64

=== SAMPLE: First 5 speeches (speaker, country, word count) ===
  Mr. De Rivière .................................. | France                              | 262 words
  The President                  |                                     | 246 words
  The President                  |                                     | 180 words
  Ms. DiCarlo. Food insecurity has continued only to worsen in recent
days. We must scale up the response.
Ms. DiCarlo |                                     | 1220 words
  The President                  |                                     | 39 words

=== ALL UNIQUE SPEAKERS ===
  The President                  | 
  Ms. DiCarlo. Food insecurity has continued 

In [10]:
# CELL 10: Full unique speakers list, sorted by document

for doc_id in ["S/PV.8812", "S/PV.8843", "S/PV.8875", "S/PV.8899"]:
    subset = df_speeches[df_speeches["document_id"] == doc_id]
    print(f"\n=== {doc_id} ===")
    for _, row in subset.iterrows():
        name_preview = row["speaker_name"][:50]
        print(f"  {name_preview:50s} | {row['country']:35s} | {row['word_count']} words")


=== S/PV.8812 ===
  Mr. De Rivière ..................................  | France                              | 262 words
  The President                                      |                                     | 246 words
  The President                                      |                                     | 180 words
  Ms. DiCarlo. Food insecurity has continued only to |                                     | 1220 words
  The President                                      |                                     | 39 words
  Mr. Rajasingham                                    |                                     | 1776 words
  The President                                      |                                     | 46 words
  Mrs. Thomas-Greenfield                             | United States of Tigray. Access by road and by air — which, along with
America | 670 words
  Lord Ahmad                                         | United Kingdom                      | 1018 words
  Ms. Byrn

In [11]:
# CELL 11: Clean the extracted speeches

# 1. ASSIGN COUNTRIES TO "The President"
# Each month has a different Council president
president_country = {
    "S/PV.8812": "France",        # July 2021: France
    "S/PV.8843": "India",         # August 2021: India
    "S/PV.8875": "Kenya",         # October 2021: Kenya
    "S/PV.8899": "Mexico",        # November 2021: Mexico
}

# 2. CLEAN FUNCTION
def clean_speech_row(row):
    row = row.copy()
    
    # Clean speaker name: remove trailing dots and whitespace
    name = re.sub(r'[\s.]+$', '', row["speaker_name"])
    # Remove any text after a period that looks like merged content
    name = re.split(r'\.\s+[A-Z]', name)[0]
    row["speaker_name"] = name.strip()
    
    # Clean country: remove merged text from two-column PDF
    country = row["country"]
    if country:
        # Take only text before "of" patterns that signal merged content
        # e.g. "United States of Tigray. Access..." -> need special handling
        country = re.split(r'\.\s+', country)[0]  # cut at first sentence break
        # Fix known issues
        if "United States" in country:
            country = "United States of America"
        if "United Kingdom" in country:
            country = "United Kingdom"
    row["country"] = country.strip()
    
    # Assign country to The President
    if row["speaker_name"] == "The President":
        row["country"] = president_country.get(row["document_id"], "")
    
    # Tag speaker role
    if row["speaker_name"] == "The President":
        row["role"] = "president"
    elif row["speaker_name"] in ["Ms. DiCarlo", "Mr. Rajasingham", "Mr. Griffiths",
                                  "Ms. Pobee", "Mr. Obasanjo", "Ms. Mudawi"]:
        row["role"] = "briefer"
    elif row["country"] == "Ethiopia":
        row["role"] = "invited_state"
    else:
        row["role"] = "council_member"
    
    return row

# Apply cleaning
df_clean = df_speeches.apply(clean_speech_row, axis=1)

# Show results
print("=== CLEANED SPEAKERS ===")
for doc_id in ["S/PV.8812", "S/PV.8843", "S/PV.8875", "S/PV.8899"]:
    subset = df_clean[df_clean["document_id"] == doc_id]
    print(f"\n{doc_id}:")
    for _, row in subset.iterrows():
        print(f"  {row['speaker_name']:35s} | {row['country']:30s} | {row['role']:15s} | {row['word_count']} words")

=== CLEANED SPEAKERS ===

S/PV.8812:
  Mr                                  | France                         | council_member  | 262 words
  The President                       | France                         | president       | 246 words
  The President                       | France                         | president       | 180 words
  Ms                                  |                                | council_member  | 1220 words
  The President                       | France                         | president       | 39 words
  Mr                                  |                                | council_member  | 1776 words
  The President                       | France                         | president       | 46 words
  Mrs                                 | United States of America       | council_member  | 670 words
  Lord Ahmad                          | United Kingdom                 | council_member  | 1018 words
  Ms                                  | Ireland      

In [12]:
# CELL 12: Fix the name cleaning

def clean_speaker_name(raw_name):
    """Clean speaker name while preserving title + surname."""
    name = raw_name.strip()
    
    # Remove trailing dots and spaces (e.g., "Mr. De Rivière ..........")
    name = re.sub(r'[\s.]{3,}$', '', name)  # remove 3+ trailing dots/spaces
    name = name.strip('. ')
    
    # If merged text got attached after the real name, cut at sentence boundary
    # Look for pattern: name followed by uppercase sentence start
    # e.g. "Ms. DiCarlo. Food insecurity has continued..."
    parts = re.split(r'(?<=\w)\.\s+[A-Z][a-z]', name)
    if parts:
        name = parts[0]
        # Add back the period if it was part of the title
        if not name.endswith('.') and name.startswith(('Mr', 'Ms', 'Mrs')):
            # Check if name has a period after title
            if re.match(r'^(Mr|Ms|Mrs)\s', name):
                name = name[:2] + '.' + name[2:]
    
    # Final cleanup
    name = name.strip('. ')
    
    # Ensure title period is preserved
    name = re.sub(r'^(Mr|Ms|Mrs)\b(?!\.)', r'\1.', name)
    
    return name

# Test on known problematic names
test_names = [
    "Mr. De Rivière ..................................",
    "Ms. DiCarlo. Food insecurity has continued only to worsen in recent days. We must scale up the response.",
    "Mr. Rajasingham",
    "Mrs. Thomas-Greenfield",
    "Lord Ahmad",
    "Ms. Byrne Nason",
    "Mr. Kimani",
    "The President",
    "Ms. Evstigneeva",
]

print("=== NAME CLEANING TEST ===")
for name in test_names:
    cleaned = clean_speaker_name(name)
    print(f"  {name[:60]:60s} -> {cleaned}")

=== NAME CLEANING TEST ===
  Mr. De Rivière ..................................            -> Mr.
  Ms. DiCarlo. Food insecurity has continued only to worsen in -> Ms.
  Mr. Rajasingham                                              -> Mr.
  Mrs. Thomas-Greenfield                                       -> Mrs.
  Lord Ahmad                                                   -> Lord Ahmad
  Ms. Byrne Nason                                              -> Ms.
  Mr. Kimani                                                   -> Mr.
  The President                                                -> The President
  Ms. Evstigneeva                                              -> Ms.


In [13]:
# CELL 13: Simpler name cleaning approach

def clean_speaker_name_v2(raw_name):
    """Clean speaker name by extracting the title + surname pattern."""
    name = raw_name.strip()
    
    # Handle "The President" first
    if name.startswith("The President"):
        return "The President"
    
    # Handle "Lord" or "Sir" (no period after title)
    lord_match = re.match(r'^(Lord|Sir)\s+(\S+)', name)
    if lord_match:
        return lord_match.group(0)
    
    # Handle Mr./Ms./Mrs. followed by a surname
    # Pattern: Title. Name (possibly hyphenated or multi-part)
    title_match = re.match(
        r'^(Mr\.|Ms\.|Mrs\.)\s+'           # Title with period
        r'([A-Z][a-zA-ZÀ-ÿ\-\s]+?)'       # Surname (unicode for accents)
        r'(?:\s*[.\(]|\s*$)',               # Ends at period, parenthesis, or string end
        name
    )
    if title_match:
        title = title_match.group(1)
        surname = title_match.group(2).strip()
        return f"{title} {surname}"
    
    # Fallback: try to grab title + first word after it
    fallback_match = re.match(r'^(Mr\.|Ms\.|Mrs\.)\s+(\S+)', name)
    if fallback_match:
        return f"{fallback_match.group(1)} {fallback_match.group(2)}"
    
    return name

# Test on known problematic names
test_names = [
    "Mr. De Rivière ..................................",
    "Ms. DiCarlo. Food insecurity has continued only to worsen",
    "Mr. Rajasingham",
    "Mrs. Thomas-Greenfield",
    "Lord Ahmad",
    "Ms. Byrne Nason",
    "Mr. Kimani",
    "The President",
    "Ms. Evstigneeva",
    "Mrs. Buenrostro Massieu",
    "Mr. Dai Bing",
    "Mr. Nebenzia",
    "Mr. Jürgenson",
    "Mr. Tirumurti",
    "Mr. Dang",
    "Ms. Juul",
    "Mr. Amde",
]

print("=== NAME CLEANING TEST v2 ===")
for name in test_names:
    cleaned = clean_speaker_name_v2(name)
    print(f"  {name[:60]:60s} -> {cleaned}")

=== NAME CLEANING TEST v2 ===
  Mr. De Rivière ..................................            -> Mr. De Rivière
  Ms. DiCarlo. Food insecurity has continued only to worsen    -> Ms. DiCarlo
  Mr. Rajasingham                                              -> Mr. Rajasingham
  Mrs. Thomas-Greenfield                                       -> Mrs. Thomas-Greenfield
  Lord Ahmad                                                   -> Lord Ahmad
  Ms. Byrne Nason                                              -> Ms. Byrne Nason
  Mr. Kimani                                                   -> Mr. Kimani
  The President                                                -> The President
  Ms. Evstigneeva                                              -> Ms. Evstigneeva
  Mrs. Buenrostro Massieu                                      -> Mrs. Buenrostro Massieu
  Mr. Dai Bing                                                 -> Mr. Dai Bing
  Mr. Nebenzia                                                 -> Mr. Neb

In [14]:
# CELL 14: Rebuild the cleaned dataframe with correct names

def clean_speech_row_v2(row):
    row = row.copy()
    
    # Clean speaker name
    row["speaker_name"] = clean_speaker_name_v2(row["speaker_raw"])
    
    # Clean country: remove merged text from two-column PDF
    country = row["country"]
    if country:
        country = re.split(r'\.\s+', country)[0]
        if "United States" in country:
            country = "United States of America"
        if "United Kingdom" in country:
            country = "United Kingdom"
    row["country"] = country.strip()
    
    # Assign country to The President
    if row["speaker_name"] == "The President":
        row["country"] = president_country.get(row["document_id"], "")
    
    # Assign country to known briefers
    briefer_map = {
        "Ms. DiCarlo": "UN Secretariat",
        "Mr. Rajasingham": "UN OCHA",
        "Mr. Griffiths": "UN OCHA",
        "Ms. Pobee": "UN Secretariat",
        "Mr. Obasanjo": "African Union",
        "Ms. Mudawi": "UN OCHA",
    }
    if row["speaker_name"] in briefer_map and not row["country"]:
        row["country"] = briefer_map[row["speaker_name"]]
    
    # Tag speaker role
    if row["speaker_name"] == "The President":
        row["role"] = "president"
    elif row["speaker_name"] in briefer_map:
        row["role"] = "briefer"
    elif row["country"] == "Ethiopia":
        row["role"] = "invited_state"
    else:
        row["role"] = "council_member"
    
    return row

# Apply cleaning
df_clean = df_speeches.apply(clean_speech_row_v2, axis=1)

# Show results
print("=== CLEANED DATASET ===\n")
for doc_id in ["S/PV.8812", "S/PV.8843", "S/PV.8875", "S/PV.8899"]:
    subset = df_clean[df_clean["document_id"] == doc_id]
    print(f"{doc_id}:")
    for _, row in subset.iterrows():
        print(f"  {row['speaker_name']:30s} | {row['country']:30s} | {row['role']:15s} | {row['word_count']} words")
    print()

print(f"Total: {len(df_clean)} speeches")
print(f"Unique speakers: {df_clean['speaker_name'].nunique()}")
print(f"Unique countries: {df_clean['country'].nunique()}")

=== CLEANED DATASET ===

S/PV.8812:
  Mr. De Rivière                 | France                         | council_member  | 262 words
  The President                  | France                         | president       | 246 words
  The President                  | France                         | president       | 180 words
  Ms. DiCarlo                    | UN Secretariat                 | briefer         | 1220 words
  The President                  | France                         | president       | 39 words
  Mr. Rajasingham                | UN OCHA                        | briefer         | 1776 words
  The President                  | France                         | president       | 46 words
  Mrs. Thomas-Greenfield         | United States of America       | council_member  | 670 words
  Lord Ahmad                     | United Kingdom                 | council_member  | 1018 words
  Ms. Byrne Nason                | Ireland                        | council_member  | 1088 words
  

In [16]:
# CELL 15: Add supplementary documents to the corpus

supplementary = []

# SG Press Releases - match by partial filename
sg_dates = {
    "20563": "2021-02-02",
    "20798": "2021-06-25",
    "20799": "2021-06-26",
    "20804": "2021-06-28",
    "20821": "2021-07-09",
    "21109": "2022-01-19",
    "21232": "2022-04-06",
    "21444": "2022-09-11",
    "21534": "2022-10-15",
    "21566": "2022-11-02",
}

for filename, text in docx_texts.items():
    if filename.startswith("SG"):
        # Extract the SG/SM number from filename
        for num, date in sg_dates.items():
            if num in filename:
                supplementary.append({
                    "document_id": f"SG/SM/{num}",
                    "date": date,
                    "speaker_raw": "Secretary-General",
                    "speaker_name": "Secretary-General",
                    "country": "UN Secretary-General",
                    "word_count": len(text.split()),
                    "text": text,
                    "role": "secretary_general",
                })
                break
    elif filename.startswith("SC"):
        supplementary.append({
            "document_id": "SC/14501",
            "date": "2021-04-22",
            "speaker_raw": "Security Council",
            "speaker_name": "Security Council",
            "country": "Security Council",
            "word_count": len(text.split()),
            "text": text,
            "role": "council_statement",
        })

# Letters from Eritrea
for filename in ["S_2021_378-EN.pdf", "S_2021_510-EN.pdf"]:
    if filename in pdf_texts:
        num = filename.split("_")[1]
        date = "2021-04-16" if "378" in filename else "2021-05-27"
        supplementary.append({
            "document_id": f"S/2021/{num}",
            "date": date,
            "speaker_raw": "Eritrea",
            "speaker_name": "Eritrea",
            "country": "Eritrea",
            "word_count": len(pdf_texts[filename].split()),
            "text": pdf_texts[filename],
            "role": "letter",
        })

print(f"Supplementary documents added: {len(supplementary)}")
for doc in supplementary:
    print(f"  {doc['document_id']:20s} | {doc['date']} | {doc['speaker_name']:25s} | {doc['word_count']} words")

Supplementary documents added: 13
  SC/14501             | 2021-04-22 | Security Council          | 334 words
  SG/SM/20563          | 2021-02-02 | Secretary-General         | 178 words
  SG/SM/20798          | 2021-06-25 | Secretary-General         | 128 words
  SG/SM/20799          | 2021-06-26 | Secretary-General         | 89 words
  SG/SM/20804          | 2021-06-28 | Secretary-General         | 86 words
  SG/SM/20821          | 2021-07-09 | Secretary-General         | 148 words
  SG/SM/21109          | 2022-01-19 | Secretary-General         | 309 words
  SG/SM/21232          | 2022-04-06 | Secretary-General         | 123 words
  SG/SM/21444          | 2022-09-11 | Secretary-General         | 174 words
  SG/SM/21534          | 2022-10-15 | Secretary-General         | 98 words
  SG/SM/21566          | 2022-11-02 | Secretary-General         | 264 words
  S/2021/2021          | 2021-04-16 | Eritrea                   | 504 words
  S/2021/2021          | 2021-05-27 | Eritrea            

In [17]:
# CELL 16: Fix Eritrea doc IDs and build final corpus

# Fix the Eritrea document IDs
for doc in supplementary:
    if doc["date"] == "2021-04-16" and doc["speaker_name"] == "Eritrea":
        doc["document_id"] = "S/2021/378"
    elif doc["date"] == "2021-05-27" and doc["speaker_name"] == "Eritrea":
        doc["document_id"] = "S/2021/510"

# Combine speeches + supplementary into one corpus
df_supplementary = pd.DataFrame(supplementary)
df_corpus = pd.concat([df_clean, df_supplementary], ignore_index=True)

# Ensure date is proper datetime
df_corpus["date"] = pd.to_datetime(df_corpus["date"])

# Sort by date
df_corpus = df_corpus.sort_values("date").reset_index(drop=True)

# Add a unique ID for each text unit
df_corpus.insert(0, "text_id", range(1, len(df_corpus) + 1))

# Summary
print(f"=== FINAL CORPUS ===")
print(f"Total text units: {len(df_corpus)}")
print(f"Total words: {df_corpus['word_count'].sum():,}")
print(f"Unique speakers: {df_corpus['speaker_name'].nunique()}")
print(f"Unique countries: {df_corpus['country'].nunique()}")
print(f"Date range: {df_corpus['date'].min().date()} to {df_corpus['date'].max().date()}")
print(f"\nBy role:")
print(df_corpus.groupby("role")["word_count"].agg(["count", "sum"]).rename(columns={"count": "texts", "sum": "words"}))
print(f"\nBy document type:")
print(df_corpus["document_id"].value_counts().head(10))

=== FINAL CORPUS ===
Total text units: 88
Total words: 51,371
Unique speakers: 33
Unique countries: 20
Date range: 2021-02-02 to 2022-11-02

By role:
                   texts  words
role                           
briefer                4   4984
council_member        48  33307
council_statement      1    334
invited_state          5   5312
letter                 2   1315
president             18   4522
secretary_general     10   1597

By document type:
document_id
S/PV.8812      20
S/PV.8899      20
S/PV.8875      18
S/PV.8843      17
SG/SM/20563     1
S/2021/378      1
SC/14501        1
S/2021/510      1
SG/SM/20798     1
SG/SM/20799     1
Name: count, dtype: int64


In [18]:
# CELL 17: Save the corpus to CSV

# Select columns to save (exclude the full text from the preview)
columns_to_save = ["text_id", "document_id", "date", "speaker_name", 
                   "country", "role", "word_count", "text"]

df_corpus[columns_to_save].to_csv("data/processed/corpus.csv", index=False)

print("Corpus saved to data/processed/corpus.csv")
print(f"File size: {os.path.getsize('data/processed/corpus.csv') / 1024:.1f} KB")

# Also save a version without text for quick inspection
df_corpus[columns_to_save[:-1]].to_csv("data/processed/corpus_metadata.csv", index=False)
print("Metadata saved to data/processed/corpus_metadata.csv")

# Quick preview of what the CSV looks like
print("\nFirst 5 rows (without text column):")
print(df_corpus[["text_id", "document_id", "date", "speaker_name", "country", "role", "word_count"]].head(10).to_string())

Corpus saved to data/processed/corpus.csv
File size: 329.2 KB
Metadata saved to data/processed/corpus_metadata.csv

First 5 rows (without text column):
   text_id  document_id       date       speaker_name               country               role  word_count
0        1  SG/SM/20563 2021-02-02  Secretary-General  UN Secretary-General  secretary_general         178
1        2   S/2021/378 2021-04-16            Eritrea               Eritrea             letter         504
2        3     SC/14501 2021-04-22   Security Council      Security Council  council_statement         334
3        4   S/2021/510 2021-05-27            Eritrea               Eritrea             letter         811
4        5  SG/SM/20798 2021-06-25  Secretary-General  UN Secretary-General  secretary_general         128
5        6  SG/SM/20799 2021-06-26  Secretary-General  UN Secretary-General  secretary_general          89
6        7  SG/SM/20804 2021-06-28  Secretary-General  UN Secretary-General  secretary_general     